In [1]:
import glob
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Get gene trait associations
RAP_DIR = 'project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = '/home/dnanexus/data_dir/'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = 'regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'
# !dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

# loftee_corrs = (
#     pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
#     .with_columns(
#         loftee_corr = pl.col('correlation'),
#         loftee_corr_abs = pl.col('correlation').abs(),
#         loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
#     )
#     .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
# )

# gene_trait_df = (
#     gene_trait_df
#     .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
#     .drop_nans()
#     .sort('loftee_corr_abs', descending=True)
#     .unique(subset=["region"], keep="first", maintain_order=True)
# )
gene_trait_df

Error: path "/home/dnanexus/data_dir/regenie_127phenotypes_lofteeHC_mac20_EUR_
miss20per.parquet" already exists but -f/--overwrite was not set


region,phenotype,pval_fdr
str,str,f64
"""ENSG00000132855""","""apolipoprotein_a_int""",0.000007
"""ENSG00000052841""","""apolipoprotein_a_int""",0.038502
"""ENSG00000110243""","""apolipoprotein_a_int""",0.003376
"""ENSG00000118137""","""apolipoprotein_a_int""",4.5099e-46
"""ENSG00000173064""","""apolipoprotein_a_int""",0.006545
…,…,…
"""ENSG00000182095""","""forced_expiratory_volume_in_1s…",0.02095
"""ENSG00000164741""","""forced_expiratory_volume_in_1s…",0.036208
"""ENSG00000205189""","""forced_expiratory_volume_in_1s…",0.045581


In [3]:
# EUR unrelated individuals

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/sample_lists/unrelated_cauc_samples_3rd_degree.csv -o /home/dnanexus/data_dir/

unrel_eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv')['eid'].cast(pl.Utf8).to_list()
unrel_eur_samples[:5]

Error: path "/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv"
already exists but -f/--overwrite was not set


['1000020', '1000107', '1000161', '1000172', '1000221']

In [4]:
from scipy.special import ndtri

c = 3/8  # Blom's constant for inverse normal transformation (prevents infinite values at the tails)

# Download phenotypes: covariates and PRS corrected
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/phenotypes/corrected_cov_PRS_traits_EUR.parquet -o /home/dnanexus/data_dir/

phenos = (
    pl.read_parquet('/home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquet')
    .rename({'individual':'sample'})
    .filter(pl.col('sample').is_in(unrel_eur_samples))
)

# phenos
long_phenos_int = (
    phenos
    .unpivot(
        index='sample',
        on=gene_trait_df['phenotype'].unique().to_list(),
        variable_name='phenotype',
        value_name='pheno_value'
    )
    .drop_nulls()

    .lazy()  # Use Lazy mode for better memory/query optimization
    .with_columns(
        # Calculate rank and group size using native Rust engine
        r = pl.col("pheno_value").rank().over("phenotype"),
        n = pl.len().over("phenotype")
    )
    .with_columns(
        # Calculate the INT value calling ndtri ONCE on the whole column
        pheno_value_int = ((pl.col("r") - c) / (pl.col("n") - 2*c + 1)).map_batches(ndtri)
    )
    .drop(["r", "n"]) # Clean up temporary columns
    .collect()
)

print(long_phenos_int['phenotype'].value_counts(sort=True))
long_phenos_int

[===========================================================>] Completed 1,717,336,582 of 1,717,336,582 bytes (100%) /home/dnanexus/data_dir/corrected_cov_PRS_traits_EUR.parquett
shape: (102, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u64    │
╞═════════════════════════════════╪════════╡
│ townsend_deprivation_index_at_… ┆ 378461 │
│ waist_circumference_int         ┆ 378281 │
│ hip_circumference_int           ┆ 378244 │
│ standing_height_int             ┆ 378108 │
│ weight_int                      ┆ 377841 │
│ …                               ┆ …      │
│ phosphate_int                   ┆ 330147 │
│ apolipoprotein_a_int            ┆ 328787 │
│ shbg_int                        ┆ 327605 │
│ testosterone_int                ┆ 327366 │
│ direct_bilirubin_int            ┆ 307355 │
└─────────────────────────────────┴────────┘


sample,phenotype,pheno_value,pheno_value_int
str,str,f64,f64
"""1000020""","""weight_impedance_int""",-0.879798,-1.12577
"""1000107""","""weight_impedance_int""",-0.561876,-0.70402
"""1000161""","""weight_impedance_int""",-0.875212,-1.120381
"""1000172""","""weight_impedance_int""",-0.882824,-1.130018
"""1000221""","""weight_impedance_int""",-0.286211,-0.335607
…,…,…,…
"""4974782""","""trunk_fatfree_mass_int""",0.218017,0.369048
"""5956310""","""trunk_fatfree_mass_int""",-0.606271,-1.11076
"""4301443""","""trunk_fatfree_mass_int""",-0.040433,-0.076444


In [5]:
long_phenos_int.filter(pl.col('phenotype')=='standing_height_int').select(pl.col('pheno_value').std())

pheno_value
f64
0.606581


In [6]:
long_phenos_int.filter(pl.col('phenotype')=='standing_height_int').select(pl.col('pheno_value_int').std())

pheno_value_int
f64
0.999992


In [7]:
mac = 20

RAP_ANNO_DIR = "project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

# ANNO_FILE = "annotations_fillna_ukbgym.parquet"
ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DIR}/{ANNO_FILE}

id_list = (
    pl.scan_parquet(f'{LOCAL_DIR}/{ANNO_FILE}')
    .filter(
        pl.col('region').is_in(gene_trait_df.select('region').unique().to_series()),
        pl.col('mac_ukb')<=mac,
        pl.col('loftee_hc')==1,
    )
    .select('id')
    .unique()
    .collect()
)

id_list

Error: path
"/home/dnanexus/data_dir//annotations_fillna_ukbgym_with_mane.parquet" already
exists but -f/--overwrite was not set


/tmp/ipykernel_39809/1805579265.py:19: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


id
str
"""chr17:60180627:CCTGAATGGCA:C"""
"""chr19:41238595:C:T"""
"""chr4:40121730:A:T"""
"""chr19:13261463:CA:C"""
"""chr18:21798139:TG:T"""
…
"""chr10:68934899:GAT:G"""
"""chr19:34339904:C:CAG"""
"""chr7:92002069:ACT:A"""


In [ ]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(
        pl.col('gt')==1,
        pl.col('sample').is_in(unrel_eur_samples),
    )
    .join(
        id_list.lazy(),
        on='id',
        how='semi'
    )

    .collect()
)

long_gt

[===========================================================>] Completed 27,974,638,284 of 27,974,638,284 bytes (100%) /home/dnanexus/data_dir/gt_long.parquett======================================================>     ] Downloaded 25,467,813,888 of 27,974,638,284 bytes (91%) /home/dnanexus/data_dir/gt_long.parquet


In [ ]:
(
    long_phenos_int.lazy()
    # .filter(pl.col('phenotype').is_in(chunk_phenos))
    .join(
        long_gt.lazy(),
        on='sample',
        how='inner'
    )
    .group_by(['id', 'phenotype'])
    .agg(
        n_individuals = pl.len().cast(pl.Int32),
        mean_pheno_value = pl.col('pheno_value_int').mean().cast(pl.Float32),
        std_pheno_value = pl.col('pheno_value_int').std().cast(pl.Float32),
    )
)